# 🛠️ Setup — Warehouse & Retail Sales ML Pipeline

**Author:** Santiago López Blanco  
**Purpose:** Reproducibility guide — run this notebook first before any other.

---

## Requirements

| Component | Version |
|-----------|--------|
| Platform | Databricks Free Edition |
| Runtime | Databricks Runtime 15.x (Apache Spark 4.1, Python 3.12) |
| Dataset | Montgomery County Open Data (see source below) |

---


## Step 1 — Install Python dependencies

Databricks manages Spark and Delta Lake automatically.  
The libraries below must be installed manually on the cluster.

> ⚠️ Do **not** install `pyspark` — it is managed by the Databricks runtime.  
> Installing it manually will cause version conflicts.


In [0]:
# Install all project dependencies
# %pip installs into the current cluster — restart the Python kernel after running this cell
%pip install \
    pandas==2.2.2 \
    numpy==1.26.4 \
    scikit-learn==1.5.1 \
    lightgbm==4.3.0 \
    xgboost==2.0.3 \
    mlflow==2.14.1 \
    matplotlib==3.9.1

## Step 2 — Restart the Python kernel

After installing dependencies, restart the kernel so the new versions take effect.  
In Databricks: **Cluster → Restart Python**  
Or run the cell below:


In [0]:
# Restart the Python process to apply new package versions
# This is equivalent to 'Restart kernel' in Jupyter
dbutils.library.restartPython()

## Step 3 — Verify installation

Run the cell below. All checks should print ✓.  
If any check fails, re-run Step 1 and restart again.


In [0]:
# Verify that all required libraries are installed and importable
checks = {
    'pandas':       '2.2.2',
    'numpy':        '1.26.4',
    'sklearn':      '1.5.1',
    'lightgbm':     '4.3.0',
    'xgboost':      '2.0.3',
    'mlflow':       '2.14.1',
    'matplotlib':   '3.9.1',
}

all_ok = True
for lib, expected in checks.items():
    try:
        mod = __import__(lib)
        version = getattr(mod, '__version__', 'unknown')
        status = '✓' if version == expected else f'⚠ expected {expected}'
        print(f'{status}  {lib} {version}')
    except ImportError:
        print(f'✗  {lib} — NOT FOUND')
        all_ok = False

print()
print('✅ All checks passed.' if all_ok else '❌ Fix the errors above before proceeding.')

## Step 4 — Load the dataset

This project uses a **public government dataset** from Montgomery County, Maryland.

| Detail | Value |
|--------|-------|
| Source | [data.montgomerycountymd.gov](https://catalog.data.gov/dataset/warehouse-and-retail-sales) |
| Format | CSV |
| Rows | 307,645 |
| Columns | 9 |
| Last modified | May 5, 2026 |

**How to load it in Databricks:**
1. Download the CSV from the link above
2. Go to **Databricks → Data → Add Data → Upload File**
3. Upload the CSV — Databricks will create a table automatically
4. Note the full table path (e.g. `main.default.warehouse_and_retail_sales`)
5. Update the table reference in `01_bronze_ingestion.ipynb` if your path differs


## Step 5 — Run notebooks in order

Each notebook reads from the output of the previous one.  
**Do not skip steps** — the Delta tables must exist before the next notebook reads them.

| Order | Notebook | Input | Output |
|-------|----------|-------|--------|
| 1 | `01_bronze_ingestion.ipynb` | Raw CSV upload | `bronze_warehouse_sales` Delta table |
| 2 | `02_silver_transformation.ipynb` | Bronze table | `silver_warehouse_sales` Delta table |
| 3 | `03_silver_EDA.ipynb` | Silver table | Visualizations (no tables written) |
| 4 | `04_gold_layer.ipynb` | Silver table | `gold_business_metrics`, `gold_ml_features` |
| 5 | `05_ml_model.ipynb` | Gold ML features | Trained model in MLflow registry |
| 6 | `06_dashboard.ipynb` | Gold + MLflow | Business visualizations + 2025 forecast |

---

## Estimated runtime

| Notebook | Approximate time |
|----------|-----------------|
| 01 — Bronze | < 1 min |
| 02 — Silver | 1–2 min |
| 03 — EDA | 2–3 min |
| 04 — Gold | 1–2 min |
| 05 — ML Model | 5–8 min |
| 06 — Dashboard | 2–3 min |

> Times measured on Databricks Free Edition serverless compute.


---

## Troubleshooting

**`Table or view not found` error**  
→ Run the notebooks in the correct order. The required Delta table hasn't been created yet.

**`ModuleNotFoundError`**  
→ Re-run Step 1 and restart the Python kernel.

**`MLflow model not found` in notebook 06**  
→ Notebook 05 must run fully to completion — the model registry step is at the end.

---

**Author:** Santiago López Blanco  
[![LinkedIn](https://img.shields.io/badge/LinkedIn-Connect-blue?logo=linkedin)](https://www.linkedin.com/in/santiago-l%C3%B3pez-blanco-ds)
